In [ ]:
#!/usr/bin/env python3
"""VSEC mini-project: clean, analyze and visualize 2025 F1 Grand Prix results.

Run: python analysis_program.py
Input: f1_2025_grand_prix_results.csv (in the same folder)
Outputs: analysis_outputs/ cleaned data, summary tables, charts, and console_output.txt
"""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path(__file__).resolve().parent
OUT = BASE / "analysis_outputs"
OUT.mkdir(exist_ok=True)
INPUT = BASE / "f1_2025_grand_prix_results.csv"

# A. LOAD AND CLEAN WITH PANDAS
raw = pd.read_csv(INPUT)
df = raw.copy()
initial_rows, initial_cols = df.shape

# Trim accidental spaces in text columns.
text_cols = df.select_dtypes(include=["object"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Parse dates and convert intended numeric columns.
df["race_date"] = pd.to_datetime(df["race_date"], errors="coerce")
numeric_cols = [
    "season", "round", "car_number", "grid_position", "finish_position",
    "laps_completed", "race_laps_leader", "lap_completion_pct",
    "gap_to_winner_seconds", "points", "fastest_lap_rank",
    "fastest_lap_number", "grid_places_gained"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize Boolean columns after CSV import.
def to_bool(series):
    return series.astype("string").str.lower().map({
        "true": True, "false": False, "1": True, "0": False
    })

for col in ["sprint_weekend", "classified", "winner", "podium", "points_finish"]:
    if col in df.columns:
        df[col] = to_bool(df[col])

# Check one driver per race-round key.
key = ["round", "driver_code"]
exact_duplicates = int(df.duplicated().sum())
duplicate_keys_before = int(df.duplicated(key).sum())

if exact_duplicates:
    df = df.drop_duplicates()

if df.duplicated(key).any():
    df = df.drop_duplicates(key, keep="first")

# Recompute derived fields.
df["grid_places_gained"] = df["grid_position"] - df["finish_position"]
df["lap_completion_pct"] = np.where(
    df["race_laps_leader"].gt(0),
    100 * df["laps_completed"] / df["race_laps_leader"],
    np.nan
)
df["winner"] = df["finish_position"].eq(1)
df["podium"] = df["finish_position"].between(1, 3, inclusive="both")
df["points_finish"] = df["points"].gt(0)

# Preserve missing optional values as NaN; do not replace them with zero.
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_by_column.rename("missing_rows").to_csv(
    OUT / "missing_values_audit.csv", header=True
)
df.to_csv(OUT / "f1_2025_cleaned.csv", index=False, encoding="utf-8-sig")

# B. ANALYZE WITH NUMPY AND PANDAS
points = df["points"].to_numpy(dtype=float)
valid_points = points[~np.isnan(points)]

mean_points = float(np.mean(valid_points))
median_points = float(np.median(valid_points))
std_points = float(np.std(valid_points, ddof=0))
q1, q3 = np.quantile(valid_points, [0.25, 0.75])

valid_grid_finish = (
    df[["grid_position", "finish_position"]]
    .dropna()
    .to_numpy(dtype=float)
)
correlation = float(
    np.corrcoef(valid_grid_finish[:, 0], valid_grid_finish[:, 1])[0, 1]
)

by_driver = (
    df.groupby(["driver", "driver_code", "team"], as_index=False)
    .agg(
        points=("points", "sum"),
        wins=("winner", "sum"),
        podiums=("podium", "sum"),
        starts=("round", "nunique"),
        mean_finish=("finish_position", "mean"),
        mean_grid=("grid_position", "mean"),
        classified_finishes=("classified", "sum"),
        fastest_laps=("fastest_lap_rank", lambda s: int((s == 1).sum()))
    )
    .sort_values(["points", "wins", "podiums"], ascending=False)
)

by_team = (
    df.groupby("team", as_index=False)
    .agg(
        points=("points", "sum"),
        wins=("winner", "sum"),
        podiums=("podium", "sum"),
        entries=("driver_code", "count")
    )
    .sort_values("points", ascending=False)
)

by_race = (
    df.groupby(["round", "grand_prix"], as_index=False)
    .agg(
        entries=("driver_code", "count"),
        classified_entries=("classified", "sum"),
        winner=("driver", lambda s: s.iloc[0]),
        race_points=("points", "sum")
    )
    .sort_values("round")
)

by_driver.to_csv(OUT / "driver_summary.csv", index=False)
by_team.to_csv(OUT / "team_summary.csv", index=False)
by_race.to_csv(OUT / "race_summary.csv", index=False)

# Validate key properties.
assert df["round"].nunique() == 24, "Expected 24 Grand Prix rounds."
assert not df.duplicated(key).any(), "Duplicate race-entry keys remain."
assert df["points"].ge(0).all(), "Unexpected negative points value."

# C. VISUALIZE WITH MATPLOTLIB AND SEABORN
sns.set_theme(style="whitegrid", palette="deep")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=by_driver.head(10),
    x="points",
    y="driver",
    color="#1479a8",
    ax=ax
)
ax.set(
    title="Top 10 drivers by 2025 Grand Prix points",
    xlabel="Points in Grand Prix races",
    ylabel="Driver"
)
fig.tight_layout()
fig.savefig(OUT / "driver_points.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=by_team, x="points", y="team", color="#d1495b", ax=ax)
ax.set(
    title="Constructor points from Grand Prix races",
    xlabel="Points",
    ylabel="Team"
)
fig.tight_layout()
fig.savefig(OUT / "team_points.png", dpi=170)
plt.close(fig)

plotdata = df.dropna(subset=["grid_position", "finish_position"])
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=plotdata,
    x="grid_position",
    y="finish_position",
    hue="points",
    palette="viridis",
    size="points",
    sizes=(12, 80),
    alpha=0.68,
    legend=False,
    ax=ax
)
ax.plot([1, 20], [1, 20], linestyle="--", color="gray")
ax.invert_xaxis()
ax.invert_yaxis()
ax.set(
    title="Starting grid vs final position",
    xlabel="Grid position (pole at left)",
    ylabel="Finish position (winner at top)"
)
fig.tight_layout()
fig.savefig(OUT / "grid_vs_finish.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=plotdata,
    x="grid_position",
    y="grid_places_gained",
    color="#8ecae6",
    showfliers=False,
    ax=ax
)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    title="Position change by starting grid",
    xlabel="Starting grid slot",
    ylabel="Grid − finish (positive = places gained)"
)
fig.tight_layout()
fig.savefig(OUT / "position_change_by_grid.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(
    data=by_race,
    x="round",
    y="classified_entries",
    color="#2a9d8f",
    ax=ax
)
ax.set(
    title="Classified entries by Grand Prix",
    xlabel="Round",
    ylabel="Classified entries",
    ylim=(0, 21)
)
fig.tight_layout()
fig.savefig(OUT / "classified_entries_by_round.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.histplot(
    data=df,
    x="points",
    bins=np.arange(-0.5, 26.5, 1),
    color="#f4a261",
    ax=ax
)
ax.set(
    title="Distribution of points per driver-race entry",
    xlabel="Race points",
    ylabel="Entry count"
)
fig.tight_layout()
fig.savefig(OUT / "points_distribution.png", dpi=170)
plt.close(fig)

# D. CREATE TEXT OUTPUT FOR THE REPORT
champ = by_driver.iloc[0]
team_lead = by_team.iloc[0]
classified_n = int(df["classified"].sum())
classified_pct = 100 * classified_n / len(df)
mean_places_gained = float(df["grid_places_gained"].mean())

lines = [
    "VSEC MINI PROJECT: 2025 FORMULA 1 DATA ANALYSIS",
    "=" * 58,
    f"Input file: {INPUT.name}",
    f"Initial dataset shape: {initial_rows} rows x {initial_cols} columns",
    f"Clean dataset shape: {df.shape[0]} rows x {df.shape[1]} columns",
    (
        f"Grand Prix rounds: {df['round'].nunique()} | "
        f"Drivers: {df['driver_code'].nunique()} | "
        f"Teams: {df['team'].nunique()}"
    ),
    f"Exact duplicates found before cleaning: {exact_duplicates}",
    f"Duplicate round-driver keys found before cleaning: {duplicate_keys_before}",
    f"Duplicate round-driver keys after cleaning: {int(df.duplicated(key).sum())}",
    (
        f"Missing cells after cleaning: {int(df.isna().sum().sum())} "
        "(optional fields preserved as missing)"
    ),
    "",
    "NUMPY POINTS STATISTICS (one driver-race record per observation)",
    (
        f"Mean: {mean_points:.3f} | Median: {median_points:.3f} | "
        f"Population SD: {std_points:.3f}"
    ),
    (
        f"Q1: {q1:.3f} | Q3: {q3:.3f} | "
        f"Min: {np.min(valid_points):.1f} | Max: {np.max(valid_points):.1f}"
    ),
    (
        "Pearson correlation: grid position vs finish position = "
        f"{correlation:.3f}"
    ),
    f"Mean places gained (grid - finish): {mean_places_gained:.3f}",
    "",
    "TOP 10 DRIVERS BY GRAND PRIX POINTS",
    by_driver.head(10)[
        ["driver", "team", "points", "wins", "podiums", "starts"]
    ].to_string(index=False),
    "",
    "TEAM POINT TOTALS (GRAND PRIX RACES ONLY)",
    by_team[["team", "points", "wins", "podiums"]].to_string(index=False),
    "",
    f"Classified records: {classified_n}/{len(df)} = {classified_pct:.2f}%",
    (
        f"Highest driver points total: {champ['driver']} "
        f"({int(champ['points'])} points)"
    ),
    (
        f"Highest team points total: {team_lead['team']} "
        f"({int(team_lead['points'])} points)"
    ),
    "",
    "Output folder: analysis_outputs/",
    (
        "Generated: cleaned CSV, missing-value audit, "
        "driver/team/race summaries, six charts."
    )
]

output_text = "\n".join(lines) + "\n"
(OUT / "console_output.txt").write_text(output_text, encoding="utf-8")
print(output_text)